#### In this notebook, we take the categorised data frame created from the player match data and competition meta data and clean it. While cleaning, we also assess and explore data quality issues.

In [1]:
from pathlib import Path
import pandas as pd
import json

In [2]:
#Getting successfully scraped files

# read project folder (1 folder up from where we are right now)
data_collection = Path("../data_collection")

# read scrape log
log = pd.read_csv(data_collection / "scrape_log.csv")

#keeping only successful logs
log_success = log[log["status"] == "success"]

# getting file NAMES from the successful logs in a list [] called success_files
success_files = log_success["file_name"].tolist()

# we have the file names of success files. Now we want the actual files, not just the names. For that, we need to define the path for python.
json_folder = data_collection / "raw_match_centres" # note that raw_match_centre is where all your downloaded JSONs are from your scraper

# getting success files and saving them in a list called records
records = []

for file in success_files: #looping on each file in success_files 
    with open(json_folder / file, "r", encoding="utf-8") as f: #adding them here
        records.append(json.load(f))

# sanity check 
len(records)
# 21,027 success records out of 23,501 total scraped records.

21027

In [3]:
# creating data frame out of it


#creating empty rows for player-match dataset
player_rows = []

#labelling the data we need for each JSON in records
for record in records: #loop through each JSON (biggest loop)
    
    # READABILITY
    
    # Give shorter names to the relevant parts of the JSON
    fixture = record["fixture"] #now fixture dictionary in each JSON is called fixture
    tournament = record["tournament"] #now events list is called events in each JSON
    events = record["events"] #now tournament dictionary is called tournament

    # Combine both teams' player lists
    players = (
        record["team1_players"]
        + record["team2_players"]
    )

    # SUBSTITUTION LOGIC
    
    # Loop through every player listed for this match/this JSON
    for player in players:

        # If any player information is empty, flag it!
        t1_empty = len(record["team1_players"]) == 0
        t2_empty = len(record["team2_players"]) == 0
        one_team_empty = t1_empty != t2_empty # exactly one of the two teams has an empty player list

        # start by assuming the player did not come on as a substitute
        substituted = False

        # Check every event in this match
        for event in events:

            # assuming a type 4 event is a substitution, a type 2 is a goal.
            # assuming aiff_player_id is the player who came on and sub_aiff_player_id is the outgoing player
            if (
                event.get("type") == 4
                and event.get("aiff_player_id") == player.get("aiff_id")
            ): #if type = 4 and id is there,
                substituted = True #players gets us all players. this marks, which of those ends up actually coming onto field. later we see that for all (almost) status 1, subt is false and for 2, substitution is true some half, re-inforcing our logic of status and type both.
                break #if you find a match of substitution for this player, break this loop, don't check more events, go to appearance loop!

        # Build one player-match row
        row = {
            "match_id": fixture.get("id"),
            "competition_id": fixture.get("tournament_id"),
            "competition_name": fixture.get("tournament_name"),
            "match_date": fixture.get("date"),
            "match_venue": fixture.get("venue"),
            "match_stadium": fixture.get("stadium"),
            "ft": fixture.get("ft"),
            "ft_text": fixture.get("ft_text"),

            "team_type": tournament.get("team_type"),

            "player_name": player.get("player_name"),
            "player_id": player.get("player_id"),
            "player_aiff_id": player.get("aiff_id"),
            "player_team_id": player.get("team_id"),
            "status": player.get("status"),
            "one_team_player_data_missing": one_team_empty,

            "substituted": substituted,
        }

        # Add this row to the list
        player_rows.append(row)

# Convert all rows into a dataframe
player_matches = pd.DataFrame(player_rows)

In [4]:
# creating team ID - team name dataset from scraped records 
team_rows = []

for record in records:
    teams_data = record.get("teams_data", {})

    # Case 1: teams_data is a dictionary
    if isinstance(teams_data, dict):

        for team_id, team_info in teams_data.items():
            team_rows.append({
                "team_id": team_id,
                "team_name": team_info.get("name"),
                "team_short_name": team_info.get("short_name"),
            })

    # Case 2: teams_data is a list
    elif isinstance(teams_data, list):

        for team_info in teams_data:
            team_rows.append({
                "team_id": team_info.get("id")
                           or team_info.get("team_id"),
                "team_name": team_info.get("name")
                             or team_info.get("team_name"),
                "team_short_name": team_info.get("short_name"),
            })

team_lookup = pd.DataFrame(team_rows)

team_lookup["team_id"] = pd.to_numeric(
    team_lookup["team_id"],
    errors="coerce"
).astype("Int64")

team_lookup = (
    team_lookup
    .dropna(subset=["team_id"])
    .drop_duplicates()
)

team_name_conflicts = (
    team_lookup
    .groupby("team_id")["team_name"]
    .nunique()
    .sort_values(ascending=False)
)

team_lookup.to_csv("team_id_name_map.csv")

In [5]:
# merging team names with dataset
player_matches = player_matches.merge(
    team_lookup[["team_id", "team_name"]],
    how="left",
    left_on="player_team_id",
    right_on="team_id",
    validate="many_to_one",
).drop(columns="team_id")

In [64]:
# combining categorisations into our dataframe. Our big dataset with scraped data and categories is called df. Our analysis dataset will be called df_analysis. 

# getting csv with competition meta data made my AI + human
comp_categories = pd.read_csv("competitions_classified_AI_Human.csv")

# Keep only the metadata columns needed for analysis from the competition metadata dataset
competition_metadata = comp_categories[
    [
        "competition_name",
        "AI_level",
        "AI_state",
        "AI_age",
        "AI_primary_format_teams",
        "AI_gender"
    ]
]

# Merge competition metadata onto player-match dataset and create new dataset called player_matches_enriched
df = player_matches.merge(
    competition_metadata,
    on="competition_name",
    how="left",
    validate="many_to_one"
)

# Check merge
print(df.shape)
df.head()

(807367, 22)


,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,player_team_id,status,one_team_player_data_missing,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,19665,2,False,True,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,19665,3,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male


In [7]:
# To confirm gender was correctly assigned.
# Show only competitions containing female-related keywords that are still labelled Male.

keywords = r"asmita|female|women|girl|girls|kanyashree"

(
    df.loc[df["AI_gender"] == "Male", "competition_name"]
      .drop_duplicates()
      .loc[
          lambda s: s.str.contains(
              keywords,
              case=False,
              na=False,
              regex=True
          )
      ]
      .sort_values()
)

7989       ASMITA FOOTBALL LEAGUE U-15 - 2024-25, BHAVNAGAR
120460         ASMITA FOOTBALL LEAGUE U-15 - 2024-25, SURAT
109685         ASMITA FOOTBALL LEAGUE U-17 - 2024-25, SURAT
467866    ASMITA Zonal Football League 2025-26,Karnataka...
503121    U -13 ASMITA Zonal Football League 2025-26, Ch...
196742    U 13 ASMITA City Football League 2025-26, Agar...
180493    U 13 ASMITA City football league 2025-26, Nara...
224401    U 13 ASMITA Football City/District League 2025...
698773    U 16 Asmita Football League 2025-26, Mancheria...
26467     U- 13 ASMITA Football League 2024- 25, Anantap...
261376    U-13 ASMITA CITY FOOTBALL LEAGUE 2025-2026 (Al...
439197    U-13 ASMITA CITY FOOTBALL LEAGUE 2025-2026 (BA...
447129    U-13 ASMITA CITY FOOTBALL LEAGUE 2025-2026 (Ja...
202664    U-13 ASMITA CITY FOOTBALL LEAGUE 2025-2026 = R...
167537    U-13 ASMITA CITY FOOTBALL LEAGUE 2025-26 BANGA...
461116                       U-13 ASMITA CITY LEAGUE SIKKIM
618960         U-13 ASMITA National Foot

In [8]:
# Manual classification overrides
# ASMITA competitions:
# Overrode AI gender classification to "Female" for competitions containing keywords such as "ASMITA", "Women's", "Girls", "Female", and "Kanyashree". These are AIFF/state women's competitions that the AI occasionally classified as Male because gender was not explicit in every competition name.

# Sanjeewan State Level U-12 and U-15 Football Tournaments (2025):
# Overrode competition format from "Club League" to "Other". Manual review confirmed these are pure knockout tournaments rather than league-format competitions, so they do not meet the study's definition of a Club League.
#manual fixes 
keywords = r"asmita|female|women|girl|girls|kanyashree"

# Manually classify obvious female competitions
df.loc[
    df["competition_name"].str.contains(
        keywords,
        case=False,
        na=False,
        regex=True
    ),
    "AI_gender"
] = "Female"

knockout_tournaments = [
    "Sanjeewan State Level U-15 Football Tournament 2025",
    "Sanjeewan State Level U-12 Football Tournament 2025",
]

df.loc[
    df["competition_name"].isin(knockout_tournaments),
    "AI_primary_format_teams"
] = "Other"

df.loc[
    df["competition_name"].isin([
        "Stay Your Age Cup - 2025",
        "JSW YOUTH CUP 2025 (UNDER -18)"
    ]),
    "AI_primary_format_teams"
] = "Other"

df.to_csv("scraped_categorised_data.csv")
len(df)

807367

In [9]:
# creating analysis dataset with Club Leagues, Male, Youth
df_analysis = df[
    (df["AI_primary_format_teams"] == "Club League") &
    (df["AI_gender"] == "Male") &
    (df["AI_age"] == "Youth")
].copy()

print(df_analysis.shape)
df_analysis.head()

(291844, 22)


,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,player_team_id,status,one_team_player_data_missing,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,19665,2,False,True,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,19665,3,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,19665,1,False,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male


In [66]:
df_analysis['AI_state'].value_counts()

AI_state
National            99871
West Bengal         31052
Goa                 29007
Maharashtra         24854
Karnataka           22811
Gujarat             16875
Delhi               12956
Kerala              11931
Rajasthan           10368
Himachal Pradesh     9250
Puducherry           7645
Assam                6886
Madhya Pradesh       2843
Punjab               1694
Chhattisgarh         1382
Odisha               1319
Manipur              1029
Tamil Nadu             41
Name: count, dtype: int64

### Cleaning and EDA

#### Some things to check
* Why is there a discrepancy between scraped records and successful records? - DONE
* Does each player have an AIFF ID? The ones that don't have an AIFF ID will have to be dropped. - DONE
* Check aiff_id and player_id differences - DONE
* For the entries where one_team_player_data_missing = True, do you drop them? - DONE
* Does each AIFF ID have one player attached to it? The ones which have more than one need to be checked. - DONE
* Does each player map to one team? The players mapping to multiple teams need to be checked. - DONE
* Confirm meaning of status and type - DONE
* check if competition_id and competition_name correspond 1:1 (one id can't have multiple names and vice versa)
* check dates format - DONE
* check if team_type is always == 2 - DONE

#### Why is there a discrepancy between scraped records and successful records?

In [10]:
print(f"Unique matches scraped successfully: {len(records)}")
print(f"Unique match IDs in data frame created out of successfully scraped matches: {len(player_matches['match_id'].value_counts())}")
print(f"Difference in successfully scraped records and data frame of successfully scraped records: {len(records) - len(player_matches['match_id'].value_counts())}")
# 25301 attempted. 21027 matches scraped successfully. 18205 remained in dataframe at player-match level due to missing both team's player information in 2822 matches.

Unique matches scraped successfully: 21027
Unique match IDs in data frame created out of successfully scraped matches: 18205
Difference in successfully scraped records and data frame of successfully scraped records: 2822


In [11]:
# Is it that the matches with no player data are responsible for the discrepancy in match counts? As player_match dataset created by looping over combined player lists from both teams in a match, where both teams have missing player data, those matches won't be accounted for.abs

both_missing = 0
one_missing = 0
neither_missing = 0

for record in records:
    t1 = record.get("team1_players", [])
    t2 = record.get("team2_players", [])

    t1_empty = len(t1) == 0
    t2_empty = len(t2) == 0

    if t1_empty and t2_empty:
        both_missing += 1
    elif t1_empty or t2_empty:
        one_missing += 1
    else:
        neither_missing += 1

print(f"Both team player lists missing: {both_missing} matches have both team players list empty")
print(f"Only one team player list missing: {one_missing} matches have one team players list empty")
print(f"Neither team player list missing: {neither_missing} matches have both team players list filled")
print(f"Total records checked: {both_missing + one_missing + neither_missing}")
# so it is clear that the matches that got dropped when created data frame from scraped records are the ones which don't have player data in them. While these matches may contain player information in the squad fields, squad data cannot be used as a substitute because it does not reliably distinguish the matchday lineup, substitutes, and players who actually appeared.

Both team player lists missing: 2822 matches have both team players list empty
Only one team player list missing: 669 matches have one team players list empty
Neither team player list missing: 17536 matches have both team players list filled
Total records checked: 21027


In [12]:
df_analysis['one_team_player_data_missing'].value_counts()

one_team_player_data_missing
False    284748
True       7096
Name: count, dtype: int64

* Out of the successfully scraped 21027 records, 13% of the records (2822) did not have player information and so had to be dropped. 
* So, about 13% of successfully scraped matches lack player-level data.
* A further 669 matches contain player data for only one team. These matches are naturally retained in the data frame, but players from the team with missing data are necessarily excluded, potentially leading to an undercount of appearances for those players.
* If this analysis is pursued for publication, it will be important to determine whether the missingness is concentrated in particular competitions or broadly distributed across the 
dataset, and whether it could systematically bias the final results. Squad lists were not used to backfill missing player lists because they did not provide the identifiers needed to link players reliably to substitution events. Treating squad-list inclusion as an appearance would risk counting unused or non-playing squad members and therefore overestimating player appearances.

#### Are there missing values for key fields?

In [13]:
# What are the fields?
df_analysis.columns
# we definitely need player_aiff_id, match_date, team_type and status

Index(['match_id', 'competition_id', 'competition_name', 'match_date',
       'match_venue', 'match_stadium', 'ft', 'ft_text', 'team_type',
       'player_name', 'player_id', 'player_aiff_id', 'player_team_id',
       'status', 'one_team_player_data_missing', 'substituted', 'team_name',
       'AI_level', 'AI_state', 'AI_age', 'AI_primary_format_teams',
       'AI_gender'],
      dtype='str')

In [14]:
# Count missing values in every column

missing_values = (
    df_analysis
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_values)

match_stadium                   123031
match_id                             0
player_team_id                       0
AI_primary_format_teams              0
AI_age                               0
AI_state                             0
AI_level                             0
team_name                            0
substituted                          0
one_team_player_data_missing         0
status                               0
player_aiff_id                       0
competition_id                       0
player_id                            0
player_name                          0
team_type                            0
ft_text                              0
ft                                   0
match_venue                          0
match_date                           0
competition_name                     0
AI_gender                            0
dtype: int64


* The only field containing missing values is match_stadium. As stadium information is not used in the appearance analysis, these missing values do not affect the analytical dataset.
* All player rows contain a non-missing AIFF ID; no rows need to be excluded on this basis.

#### Does each AIFF ID have one player attached to it? The ones which have more than one need to be checked.

In [15]:
# checking # of players per id 
unique_players_per_id = df_analysis.groupby("player_aiff_id")["player_name"].nunique().sort_values(ascending=False)
unique_players_per_id
# so multiple unique player names are attached to one player aiff id

player_aiff_id
342710    4
292057    4
209747    4
304215    3
208503    3
         ..
325619    1
325523    1
325492    1
325490    1
529901    1
Name: player_name, Length: 29568, dtype: int64

In [16]:
# checkign only for entries where player per id > 1 
more_than_one_player_per_aiff_id = unique_players_per_id[unique_players_per_id>1]
more_than_one_player_per_aiff_id
with pd.option_context("display.max_rows", None): #display all rows
    display(more_than_one_player_per_aiff_id)

player_aiff_id
342710    4
292057    4
209747    4
304215    3
208503    3
207089    3
172869    3
388844    3
381593    3
341727    3
389980    3
296143    3
285400    3
208546    3
255778    3
254474    3
360113    3
268886    3
285247    3
269068    3
253772    3
253756    3
262391    3
208547    3
253692    3
209819    3
171227    3
210442    3
171109    3
370625    3
211180    3
171417    3
171435    3
289278    3
187694    3
278857    3
212273    3
209748    3
385038    3
385157    3
171803    3
289860    3
171877    3
214273    3
253723    3
187687    3
187986    3
346151    3
187312    3
186751    3
187321    3
285166    3
186441    3
188221    3
284670    3
284398    3
284751    3
284752    3
189777    3
284176    3
187477    3
352761    3
351995    3
413037    3
393981    3
171801    2
171606    2
171649    2
171881    2
410785    2
171879    2
187552    2
171878    2
171827    2
187539    2
171806    2
171807    2
171601    2
171813    2
171816    2
182879    2
171556    2
1

In [17]:
# getting # of IDs for which repetition occurs 
more_than_one_player_per_aiff_id.value_counts().sort_index()

player_name
2    529
3     62
4      3
Name: count, dtype: int64

* 539 AIFF IDs have 2 players attached to them.
* 62 AIFF IDs have 3 players attached to them.
* 3 AIFF IDs have 4 players attached to them.

In [18]:
# inspecting the repetitions 
aiff_id_name_check = (
    df_analysis[
        df_analysis["player_aiff_id"].isin(more_than_one_player_per_aiff_id.index)
    ]
    .groupby("player_aiff_id")["player_name"]
    .agg(lambda x: sorted(x.unique()))
    .reset_index(name="player_names")
)

pd.set_option("display.max_colwidth", None)
aiff_id_name_check.head(30)

,player_aiff_id,player_names
0,48902,"[Rayaan Shakeer, Rayaan Shakeer (loc)]"
1,49943,"[Ahan Gupta, Ahan Gupta (loc)]"
2,56789,"[Sarath Rajkumar (loc), Sarath Rajkumar (u23)]"
3,56793,"[Essa Mohamed Yenepoya, Essa Mohamed Yenepoya (loc)]"
4,56798,"[Vignesh Vinod, Vignesh Vinod (loc)]"
5,67845,"[Prasanna Swaminathan, Prasanna Swaminathan (loc)]"
6,77055,"[Yohann Victor Britto, Yohann Victor Britto (loc)]"
7,77480,"[Mallikjan Kalegar, Mallikjan Kalegar (u23)]"
8,82431,"[Udit G, Udit G (loc)]"
9,90442,"[Sujay S (loc), Sujay S (u23)]"


In [19]:
# checking what bracket text usually contains 
bracket_text = (
    df_analysis["player_name"]
    .str.extract(r"\(([^()]*)\)", expand=False)
    .str.strip()
    .str.lower()
)

bracket_text.value_counts(dropna=False)

player_name
NaN                     271414
loc                      14687
.                         4328
non-loc                    633
non-local                  182
l                          178
nl                          73
local                       67
pass                        63
p                           58
u23                         45
u-23                        30
non- loc                    16
                            15
osp                         11
*                            9
fail                         7
not eligible anymore         6
non local                    5
5 match ban                  5
pundalik                     4
non- local                   2
f                            2
na                           2
lc                           1
ineligible                   1
Name: count, dtype: int64

In [20]:
# checking pundalik: can we just remove (pundalik)?
df_analysis[
    df_analysis["player_name"].str.contains(
        r"\(pundalik\)",
        case=False,
        regex=True,
        na=False
    )
][["player_aiff_id", "player_name"]]

,player_aiff_id,player_name
92631,409571,Nihal (pundalik) Nilesh Shetye
213073,409571,Nihal (pundalik) Nilesh Shetye
213163,409571,Nihal (pundalik) Nilesh Shetye
213251,409571,Nihal (pundalik) Nilesh Shetye


In [21]:
# Remove administrative annotations from player names and standardize formatting.
# Note: '(pundalik)' is intentionally preserved (if we remove it, we could remove important name identifiers).

import re

annotations_to_remove = [
    ".",
    "loc",
    "local",
    "l",
    "lc",
    "nl",
    "non-loc",
    "non-local",
    "non- loc",
    "non local",
    "non- local",
    "pass",
    "p",
    "u23",
    "u-23",
    "osp",
    "*",
    "fail",
    "not eligible anymore",
    "5 match ban",
    "f",
    "na",
    "ineligible",
]

annotation_pattern = "|".join(
    re.escape(annotation) for annotation in annotations_to_remove
)

df_analysis["player_name_clean"] = (
    df_analysis["player_name"]
    # Remove specified annotations when enclosed in parentheses.
    .str.replace(
        rf"\s*\(\s*(?:{annotation_pattern})\s*\)",
        "",
        regex=True,
        flags=re.IGNORECASE,
    )
    # Remove trailing administrative labels not enclosed in parentheses.
    .str.replace(
        r"\s*-?\s*(?:p|loc|nl|non-local|pass)\s*$",
        "",
        regex=True,
        case=False,
    )
    # Remove trailing punctuation.
    .str.replace(r"[\s.\-*]+$", "", regex=True)
    # Standardize whitespace.
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [22]:
# checkign only for entries where player per id > 1 again
unique_players_per_id = df_analysis.groupby("player_aiff_id")["player_name_clean"].nunique().sort_values(ascending=False)
more_than_one_player_per_aiff_id = unique_players_per_id[unique_players_per_id>1]
more_than_one_player_per_aiff_id.value_counts().sort_index()
# now only 32 aiff id's have 2 player names attached to them.

player_name_clean
2    32
Name: count, dtype: int64

In [62]:
# inspecting the repetitions 
aiff_id_name_check = (
    df_analysis[
        df_analysis["player_aiff_id"].isin(more_than_one_player_per_aiff_id.index)
    ]
    .groupby("player_aiff_id")["player_name_clean"]
    .agg(lambda x: sorted(x.unique()))
    .reset_index(name="player_names")
)

pd.set_option("display.max_colwidth", None)
aiff_id_name_check.head(100)

,player_aiff_id,player_names


In [24]:
# dealing with non-standardized name discrepancies: for each listed player_aiff_id, use this one standardized name

player_name_corrections = {
    132176: "Jess Sanjay Bhobe", #every row with player_aiff_id == 132176 should have player_name_clean changed to "Jess Sanjay Bhobe".
    162111: "Wangkheirakpam Punshiba Meetei",
    170173: "Muhamad Anas U R",
    177978: "Houlungou Mate",
    212431: "Tanishq Harshal Pandya",
    214458: "Devanandhan K S",
    239436: "Lalremruatfela R",
    246137: "Chetan Konthoujam Singh",
    248339: "Prisha Bhairav Patel",
    264022: "Trishan Naskar",
    265482: "Abhisekh Mahato",
    275530: "Raja Baidya",
    284251: "Devyaan Anand Shah",
    284751: "Vihaan Pandey",
    284754: "Ch Sakip",
    289637: "Marshal Murmu",
    289860: "Dhaanya Harshit Desai",
    323162: "Ahamed Kaaf Bin Firos",
    323500: "Mehtab Singh",
    325216: "Faizal Hussain",
    325575: "Ajilesh S Babu",
    362253: "Nishant Mangesh Tawade",
    382630: "Martin Janggoulen Khongsai",
    409556: "Mahendra Prakash Naik",
    410785: "Sawan Jha",
    425876: "Sayan Mondal",
    426895: "Maxwell Vailan Silva",
    430458: "Shashwat Kamboj",
    430887: "Reya Rakesh Kumar Maharajwala",
    433733: "Pahal Nabin Agarwal",
    439677: "Bikram Lohra",
    475548: "Soham Tiwari",
    513252: "Gulshan Kh Singha",
    491665: "Damanpreet Kaur",
}

# apply the corrections 
df_analysis["player_name_clean"] = (
    df_analysis["player_aiff_id"]
    .map(player_name_corrections)
    .fillna(df_analysis["player_name_clean"]) #where there is no manual correction, retain the existing cleaned name.
)

In [25]:
# checkign only for entries where player per id > 1 again
unique_players_per_id = df_analysis.groupby("player_aiff_id")["player_name_clean"].nunique().sort_values(ascending=False)
more_than_one_player_per_aiff_id = unique_players_per_id[unique_players_per_id>1]
more_than_one_player_per_aiff_id.value_counts().sort_index()

player_name_clean
2    2
Name: count, dtype: int64

In [26]:
# inspecting the repetitions 
aiff_id_name_check = (
    df_analysis[
        df_analysis["player_aiff_id"].isin(more_than_one_player_per_aiff_id.index)
    ]
    .groupby("player_aiff_id")["player_name_clean"]
    .agg(lambda x: sorted(x.unique()))
    .reset_index(name="player_names")
)
aiff_id_name_check

,player_aiff_id,player_names
0,208690,"[Kiyan Lalwani, Vansh Lalwani]"
1,401413,"[Yash Dhaval, Yash Sharma]"


In [27]:
# is player id also the same for these? 
df_analysis[
    df_analysis["player_aiff_id"].isin([208690, 401413])
][[
    "player_aiff_id",
    "player_id",
    "player_name",
    "player_team_id",
    "competition_name",
    "match_date"
]].sort_values(
    ["player_aiff_id", "match_date"]
)
# seems like a data entry errors. player_id different, aiff id attached same by mistake.

,player_aiff_id,player_id,player_name,player_team_id,competition_name,match_date
497102,208690,399897,Vansh Lalwani,14440,GUJARAT SFA (U-15) BOYS YOUTH LEAGUE - 2025-26,"01 February 2026,"
465274,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"02 April 2026,"
464657,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"02 February 2026,"
296329,208690,399897,Vansh Lalwani,14440,GUJARAT SFA (U-15) BOYS YOUTH LEAGUE - 2025-26,"02 November 2025,"
514606,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"03 April 2026,"
465038,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"04 April 2026,"
409800,208690,399897,Vansh Lalwani,14440,GUJARAT SFA (U-15) BOYS YOUTH LEAGUE - 2025-26,"04 January 2026,"
514806,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"05 April 2026,"
252974,208690,399897,Vansh Lalwani,14440,GUJARAT SFA (U-15) BOYS YOUTH LEAGUE - 2025-26,"05 October 2025,"
465573,208690,396223,Kiyan Lalwani,14440,AIFF Sub Junior League 2025-26,"07 April 2026,"


In [28]:
# Create a cleaned player identifier, initially identical to the original AIFF ID.
df_analysis["player_aiff_id_cleaned"] = (
    df_analysis["player_aiff_id"].astype(str)
)

# AIFF IDs manually verified to refer to two different players.
collision_ids = [208690, 401413]

# Identify rows corresponding to these conflicting AIFF IDs.
mask = df_analysis["player_aiff_id"].isin(collision_ids)

# For conflicting cases, append player_id to create unique player identifiers
# (e.g. 208690_396223 and 208690_399897).
df_analysis.loc[mask, "player_aiff_id_cleaned"] = ( #CREATING NEW VARIABLE PLAYER_AIFF_ID_CLEANED
    df_analysis.loc[mask, "player_aiff_id"].astype(str)
    + "_"
    + df_analysis.loc[mask, "player_id"].astype(str)
)

# Examples:
# 208690 → 208690_396223 (Kiyan Lalwani), 208690_399897 (Vansh Lalwani)
# 401413 → 401413_413668 (Yash Sharma), 401413_445711 (Yash Dhaval)

In [29]:
# checkign only for entries where player per id > 1 again
unique_players_per_id = df_analysis.groupby("player_aiff_id_cleaned")["player_name_clean"].nunique().sort_values(ascending=False)
more_than_one_player_per_aiff_id = unique_players_per_id[unique_players_per_id>1]
more_than_one_player_per_aiff_id.value_counts().sort_index()
# done!

Series([], Name: count, dtype: int64)

#### Check if competition_id and competition_name correspond 1:1 (one id can't have multiple names and vice versa)

In [30]:
competition_names_per_id = (
    df_analysis
    .groupby("competition_id")["competition_name"] # how many names attached to one competition id?
    .nunique() #how many unique names attached to one competition id?
)

competition_ids_with_multiple_names = (
    competition_names_per_id[
        competition_names_per_id > 1
    ]
)

print(
    f"Competition IDs with multiple names: "
    f"{len(competition_ids_with_multiple_names)}"
)

print(competition_names_per_id)

Competition IDs with multiple names: 0
competition_id
1014    1
1139    1
1140    1
1141    1
1203    1
       ..
2096    1
2097    1
2098    1
2112    1
2130    1
Name: competition_name, Length: 90, dtype: int64


In [31]:
# checking the reverse 
competition_ids_per_name = (
    df_analysis
    .groupby("competition_name")["competition_id"]
    .nunique()
)

competition_names_with_multiple_ids = (
    competition_ids_per_name[
        competition_ids_per_name > 1
    ]
)

print(
    f"Competition names with multiple IDs: "
    f"{len(competition_names_with_multiple_ids)}"
)

Competition names with multiple IDs: 0


* Competition name and IDs map 1:1 with each other. Competition IDs and competition names correspond one-to-one; no ID is associated with multiple names, and no name is associated with multiple IDs.

#### Does each player map to one team? The players mapping to multiple teams within the same competition / match need to be checked. (Note that multiple teams across all competitions is expected)

In [32]:
# count how many teams each player is associated with
teams_per_player = (
    df_analysis
    .groupby("player_aiff_id_cleaned")["team_name"]
    .nunique()
)

players_with_multiple_teams = (
    teams_per_player[
        teams_per_player > 1
    ]
)
print(
    f"Players associated with multiple teams: "
    f"{len(players_with_multiple_teams)}"
)

Players associated with multiple teams: 2031


In [33]:
player_team_check = (
    df_analysis[
        df_analysis["player_aiff_id_cleaned"].isin(
            players_with_multiple_teams.index
        )
    ]
    .groupby("player_aiff_id_cleaned")
    .agg(
        player_names=("player_name_clean", lambda x: sorted(x.unique())),
        teams=("team_name", lambda x: sorted(x.unique())),
    )
    .reset_index()
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(player_team_check)
#a player can show up for multiple teams. the inconsistency will be when they show up for multiple teams within the same competition/league!

,player_aiff_id_cleaned,player_names,teams
0,102656,[Sachin R],"[Kickstart FC Karnataka, Mahesh Raj United FC Bangalore]"
1,103455,[Saikhom Abinash Singh],"[Mahesh Raj United FC Bangalore, Sporting Club Delhi]"
2,105110,[Manjot Singh],"[FC Goa, Punjab FC]"
3,105384,[Alan Saji],"[FC Goa, Sporting Club Delhi]"
4,112584,[Surya Ragish],"[Alchemy International FA, Roots Football Club]"
5,115654,[Thangjam Roshan Singh],"[CLASSIC FOOTBALL ACADEMY, Mohun Bagan Super Giant]"
6,116337,[Daksh Malhotra],"[Maharashtra Oranje FC, Roots Football Club]"
7,116456,[Saish Gaunkar],"[FC Goa, SESA Football Academy]"
8,116633,[Khuraijam Rishikanta Singh],"[Kickstart FC Karnataka, Rajasthan United Football Club]"
9,117205,[Lalthangliana],"[Diamond Harbour Football Club, FC Goa]"


In [34]:
# Does a player appear for multiple team IDs within the same competition
teams_per_player_competition = (
    df_analysis
    .groupby(
        ["player_aiff_id_cleaned", "competition_id"]
    )["team_name"]
    .nunique()
)

player_competition_conflicts = (
    teams_per_player_competition[
        teams_per_player_competition > 1
    ]
)

print(
    "Player-competition combinations with multiple team names: "
    f"{len(player_competition_conflicts)}"
)

Player-competition combinations with multiple team names: 2


In [35]:
conflict_rows = (
    df_analysis
    .set_index(["player_aiff_id_cleaned", "competition_id"])
    .loc[player_competition_conflicts.index]
    .reset_index()
)

conflict_rows = conflict_rows.sort_values(
    ["player_name", "competition_name", "match_date"]
)

conflict_rows

,player_aiff_id_cleaned,competition_id,match_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,...,status,one_team_player_data_missing,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender,player_name_clean
12,387210,1781,63026,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"01 February 2026,","ARA FC, AHMEDABAD",,True,This match has finished,2,...,3,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy
16,387210,1781,64632,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"01 March 2026,","BARODA FA, VADODARA",,True,This match has finished,2,...,3,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy
3,387210,1781,57948,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"02 November 2025,","BARODA FA, VADODARA",,True,This match has finished,2,...,1,False,False,"RANGER FOOTBALL ACADEMY, BARODA",State,Gujarat,Youth,Club League,Male,Krishiv Swamy
9,387210,1781,60720,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"04 January 2026,","ARA FC, AHMEDABAD",,True,This match has finished,2,...,1,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy
1,387210,1781,56854,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"05 October 2025,","BARODA FA, VADODARA",,True,This match has finished,2,...,1,False,False,"RANGER FOOTBALL ACADEMY, BARODA",State,Gujarat,Youth,Club League,Male,Krishiv Swamy
7,387210,1781,59709,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"07 December 2025,","BARODA FA, VADODARA",,True,This match has finished,2,...,1,False,False,"RANGER FOOTBALL ACADEMY, BARODA",State,Gujarat,Youth,Club League,Male,Krishiv Swamy
13,387210,1781,63247,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"08 February 2026,","BARODA FA, VADODARA",,True,This match has finished,2,...,3,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy
17,387210,1781,64834,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"08 March 2026,","BARODA FA, VADODARA",,True,This match has finished,2,...,3,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy
4,387210,1781,58139,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"09 November 2025,","BARODA FA, VADODARA",,True,This match has finished,2,...,1,False,False,"RANGER FOOTBALL ACADEMY, BARODA",State,Gujarat,Youth,Club League,Male,Krishiv Swamy
10,387210,1781,61523,GUJARAT SFA (U-13) BOYS YOUTH LEAGUE - 2025-26,"11 January 2026,","BARODA FA, VADODARA",,True,This match has finished,2,...,3,False,False,NAVRACHANA SPORTS ACADEMY,State,Gujarat,Youth,Club League,Male,Krishiv Swamy


In [36]:
# confirming that shashwat is appearing for opposing teams (that the two teams he is seeming to appear for are actually opposing teams)
for record in records:
    fixture = record.get("fixture", {})

    if fixture.get("id") in [52177, 52181]:
        print({
            "match_id": fixture.get("id"),
            "team1_id": fixture.get("team1_id"),
            "team1_name": fixture.get("team1_name"),
            "team2_id": fixture.get("team2_id"),
            "team2_name": fixture.get("team2_name"),
        })

{'match_id': 52177, 'team1_id': 14340, 'team1_name': None, 'team2_id': 14369, 'team2_name': None}
{'match_id': 52181, 'team1_id': 14369, 'team1_name': None, 'team2_id': 14340, 'team2_name': None}


* Krishiv Swamy does not look especially problematic. He appears for team 12674 through November 30, 2025, then for team 14440 from December 28 onward. Because the change is sequential and there is no overlap, it could reflect a mid-season transfer or registration change. Keep the appearances under the same player unless another field indicates two different people.
* One player was removed because the same player identifier appeared for opposing teams in the same matches, indicating an unresolved inconsistency in the source data.

In [37]:
# Remove the player with conflicting team assignments within the same matches.
df_analysis = df_analysis[
    df_analysis["player_aiff_id_cleaned"] != "430458"
].copy()

In [38]:
# Multiple teams within the same match confirming
same_match_conflicts = (
    df_analysis
    .groupby(["player_aiff_id_cleaned", "match_id"])["team_name"]
    .nunique()
)

same_match_conflicts = same_match_conflicts[same_match_conflicts > 1]
same_match_conflicts

Series([], Name: team_name, dtype: int64)

#### Are values in status and type in line with our assumptions about those variables?

In [39]:
df_analysis['status'].value_counts()

status
1    132491
3     82972
2     74861
4       990
0       500
Name: count, dtype: int64

In [40]:
# what is status 0?
status_0 = df_analysis[df_analysis["status"] == 0]

status_0[
    [
        "match_id",
        "competition_name",
        "player_name",
        "team_name",
        "substituted",
    ]
].head(30)

,match_id,competition_name,player_name,team_name,substituted
9271,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Ritik Yadav,Penha De France Sports Club,False
9273,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Raunak Viswas Volvoikar,Penha De France Sports Club,False
9274,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Babu Hussain Sheikh,Penha De France Sports Club,False
9276,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Ananthkumar Laxman Lamani,Penha De France Sports Club,False
9280,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Tejas Nagvekar,Penha De France Sports Club,False
9284,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Hrutvik Rajendra Paryekar,Penha De France Sports Club,False
9285,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Kabeer Mulgund,Penha De France Sports Club,False
9286,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Mehfuz Halsi,Penha De France Sports Club,False
9288,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Gatikh Deepak Volvoikar,Penha De France Sports Club,False
9294,49272,GFA U-19 DIV 1 LEAGUE MENS 2024-25,Sai Rupesh Parab,Penha De France Sports Club,False


In [41]:
status_0['match_id'].value_counts()

match_id
49272    28
60045    19
57426    18
60114    18
64648    18
63438    17
59049    16
59918    15
69040    15
60431    14
60797    14
63555    14
59197    13
57445    12
57448    12
61114    12
61146    12
61149    12
69133    12
61095    11
61116    11
67089    11
57437    10
63467     9
55260     8
58898     8
59986     8
69496     8
59048     7
60782     7
60815     7
61039     7
61111     7
58214     6
60789     6
60807     6
60812     6
67900     6
55258     5
55262     5
60786     5
66686     5
68955     5
55263     4
60776     4
57855     3
57858     3
61107     3
62281     3
55259     2
57446     2
57459     2
60120     2
60803     2
60810     2
61443     2
57968     1
Name: count, dtype: int64

In [42]:
# what is status 4?
status_4 = df_analysis[df_analysis["status"] ==4]

status_4[
    [
        "match_id",
        "competition_name",
        "player_name",
        "team_name",
        "substituted",
    ]
].head(30)


,match_id,competition_name,player_name,team_name,substituted
162,70599,R-LEAGUE YOUTH U13 (Boy's) 2025-26,Virendra Singh,Jaipur Elite Football Club,False
217,70601,R-LEAGUE YOUTH U13 (Boy's) 2025-26,Aryaveer Bhargava,Jaipur Futsal,False
10188,49293,Delhi Future Stars Youth League U-19 2024/25,Raj Kumar,The Dream Team,False
10985,49329,KSFA Youth Premier League U-17 Tier-1 2024-25,Paul Pradeep,Alchemy International FA,False
11055,49330,KSFA Youth Premier League U-17 Tier-1 2024-25,Tejas Vikram (loc),Bhaichung Bhutia Football Schools- Bengaluru,False
11063,49330,KSFA Youth Premier League U-17 Tier-1 2024-25,Arvindh C (loc),Bhaichung Bhutia Football Schools- Bengaluru,False
14377,49442,Delhi Future Stars Youth League U-17 2024/25,Shaunak Ganguly,Hindustan Football Club,False
18728,49650,Delhi Future Stars Youth League U-19 2024/25,Prince Verma,The Dream Team,False
20826,49719,AIFF Junior League 2024-25,Soibam Alex Singh,Belgaum United Football Academy,False
20870,49720,AIFF Junior League 2024-25,Siddhant Chiplunkar,Mumbai City FC,False


Status values 0,3, and 4 were observed only among players who neither started nor entered as substitutes, and were therefore treated as non-appearances. we don't know what the difference is, but for the purpose of the analysis, that difference is not relevant.

In [43]:
# now, confirming if our substituted status makes sense! my hypothesis is:
# Status 1: should never be marked as substituted (because they are starters, not entrants).
# Status 2: may or may not be substituted. Some bench players come on, many don't.
# Status 3: should never be substituted.
# Status 4: should never be substituted.
# Status 0: should never be substituted.
pd.crosstab(
    df_analysis["status"],
    df_analysis["substituted"],
    normalize="index"
).round(3)
# awesome!

substituted,False,True
status,,
0,1.00,0.00
1,1.00,0.00
2,0.55,0.45
3,1.00,0.00
4,1.00,0.00


In [44]:
df_analysis["appearance"] = (
    (df_analysis["status"] == 1)
    | (df_analysis["substituted"]) # appearance is true if you either started the match OR you came on later as a substitute. Note that match hours would differ!
)

In [45]:
# Only bench players should ever be flagged as substituted on 
df_analysis[
    (df_analysis["substituted"]) &
    (df_analysis["status"] != 2)
][
    ["match_id", "player_name", "status", "team_name"]
]

,match_id,player_name,status,team_name
102973,52160,Vihaan Dhyani,1,Summerhill United Football Club
103085,52164,Vihaan Dhyani,1,Summerhill United Football Club
104991,52218,Kaushik Mani,1,BPSS FC
104998,52218,Yohaan Philip Jose,1,BPSS FC
494886,62972,Rajan Bhajo Lambor,1,Guardian Angel Sports Club
792729,71629,Gaurab Boro,1,Private Sports Academy


In [46]:
#seeing all rows with kaushik mani to confirm it doesn't impact our analysis even if 4 rows show that starter also was substitute
cols = [
    "match_id",
    "competition_name",
    "team_name",
    "player_name",
    "player_aiff_id",
    "status",
    "substituted",
    "appearance",
]

df_analysis.loc[
    df_analysis["player_name"].eq("Kaushik Mani"),
    cols
].sort_values("match_id")

,match_id,competition_name,team_name,player_name,player_aiff_id,status,substituted,appearance
86291,51705,KSFA Youth Premier League U-21 2024-25,BPSS FC,Kaushik Mani,179998,2,False,False
104991,52218,KSFA Youth Premier League U-21 2024-25,BPSS FC,Kaushik Mani,179998,1,True,True
112477,52453,KSFA Youth Premier League U-21 2024-25,BPSS FC,Kaushik Mani,179998,1,False,True
114331,52530,KSFA Youth Premier League U-21 2024-25,BPSS FC,Kaushik Mani,179998,1,False,True
116182,52574,KSFA Youth Premier League U-21 2024-25,BPSS FC,Kaushik Mani,179998,2,True,True


In [47]:
df_analysis.loc[
    df_analysis["player_aiff_id"] == 179998,
    "appearance"
].sum()
# returns 4, no over counting! so we can keep these rows. 

np.int64(4)

* Internal consistency check: A small number of player-match records (6) were internally inconsistent, with the same player recorded both as a starter (status = 1) and as an incoming substitute (aiff_player_id in a substitution event). These records were retained because both indicators independently confirm that the player appeared in the match. Since the dataset contains one row per player per match and appearances were defined as (status == 1) OR (entered as a substitute), these inconsistencies did not result in double-counting. Manual validation confirmed that affected players were counted once per match.

In [48]:
# No starter should ever be flagged as substituted on
df_analysis[(df_analysis["status"] == 1) & (df_analysis["substituted"])]

,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender,player_name_clean,player_aiff_id_cleaned,appearance
102973,52160,1430,HP U-17 Men's YOUTH League 2025,"03 April 2025,",NFL Nangal,,True,This match has finished,2,Vihaan Dhyani,...,True,Summerhill United Football Club,State,Himachal Pradesh,Youth,Club League,Male,Vihaan Dhyani,447996,True
103085,52164,1430,HP U-17 Men's YOUTH League 2025,"07 April 2025,",NFL Nangal,,True,This match has finished,2,Vihaan Dhyani,...,True,Summerhill United Football Club,State,Himachal Pradesh,Youth,Club League,Male,Vihaan Dhyani,447996,True
104991,52218,1565,KSFA Youth Premier League U-21 2024-25,"28 May 2025,",Bangalore Football Stadium,,True,This match has finished,2,Kaushik Mani,...,True,BPSS FC,State,Karnataka,Youth,Club League,Male,Kaushik Mani,179998,True
104998,52218,1565,KSFA Youth Premier League U-21 2024-25,"28 May 2025,",Bangalore Football Stadium,,True,This match has finished,2,Yohaan Philip Jose,...,True,BPSS FC,State,Karnataka,Youth,Club League,Male,Yohaan Philip Jose,188401,True
494886,62972,1947,GFA U-16 BOYS DIV 1 LEAGUE 2025-26,"29 April 2026,",Sinquetim Sports Complex,,True,This match has finished,2,Rajan Bhajo Lambor,...,True,Guardian Angel Sports Club,State,Goa,Youth,Club League,Male,Rajan Bhajo Lambor,254199,True
792729,71629,2046,"Assam Youth League U-17, 2025-26","14 May 2026,",Changsari field,,True,This match has finished,2,Gaurab Boro,...,True,Private Sports Academy,State,Assam,Youth,Club League,Male,Gaurab Boro,526103,True


#### What is team_type?

In [49]:
df_analysis['team_type'].value_counts()

team_type
2    291814
Name: count, dtype: int64

In [50]:
df_analysis.loc[
    df_analysis["team_type"] == 1,
    "competition_name"
].drop_duplicates().sort_values()

Series([], Name: competition_name, dtype: str)

#### What is the date-time column looking like?

In [51]:
df_analysis.head(200)

,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender,player_name_clean,player_aiff_id_cleaned,appearance
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Kabir Mehta,418848,True
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,True,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Tavish Sharma,398082,True
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Eklavya Choudhary,522296,False
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Swastik Sharma,398260,True
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Navya Mittal,418674,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"21 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Naman Grover,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Naman Grover,528638,False
196,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"21 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Trinay Mansinghka,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Trinay Mansinghka,425684,False
197,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"21 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Raunak Pamnani,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Raunak Pamnani,406811,True
198,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"21 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Sachin,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Sachin,528339,True


In [52]:
df_analysis["match_date"] = pd.to_datetime(
    df_analysis["match_date"],
    errors="coerce"
)

In [53]:
df_analysis["match_date"].isna().sum()

np.int64(0)

In [54]:
df_analysis.head(200)

,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender,player_name_clean,player_aiff_id_cleaned,appearance
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Kabir Mehta,418848,True
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,True,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Tavish Sharma,398082,True
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Eklavya Choudhary,522296,False
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Swastik Sharma,398260,True
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Navya Mittal,418674,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-21,\nPoornima University,NaN,True,This match has finished,2,Naman Grover,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Naman Grover,528638,False
196,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-21,\nPoornima University,NaN,True,This match has finished,2,Trinay Mansinghka,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Trinay Mansinghka,425684,False
197,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-21,\nPoornima University,NaN,True,This match has finished,2,Raunak Pamnani,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Raunak Pamnani,406811,True
198,70601,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-21,\nPoornima University,NaN,True,This match has finished,2,Sachin,...,False,Red Stone City Football Club,State,Rajasthan,Youth,Club League,Male,Sachin,528339,True


In [55]:
df_analysis["match_date"].agg(["min", "max"])
# the dataset starts from 11th October 2024 and ends on 7th July 2026.

min   2024-11-09
max   2026-07-07
Name: match_date, dtype: datetime64[us]

In [56]:
df_analysis["match_date"].dt.year.value_counts().sort_index()

match_date
2024      1738
2025    120254
2026    169822
Name: count, dtype: int64

In [57]:
#checking 2024 entries 
df_analysis.loc[
    df_analysis["match_date"].dt.year == 2024,
    ["competition_id", "competition_name", "match_date"]
].drop_duplicates().sort_values("match_date")

,competition_id,competition_name,match_date
67469,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-11-09
67742,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-11-10
67795,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-11-23
68093,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-11-24
68177,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-11-30
68383,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-12-01
68429,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-12-07
68565,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-12-08
94058,1254,GFA U-15 DIV 2 LEAGUE 2024-25,2024-12-11
68603,1204,KSFA Youth Premier League U-17 Tier-2 2024-25,2024-12-14


In [58]:
# make dataset of 2025 - 2026 season and save it for final analysis!

In [59]:
season_start = pd.Timestamp("2025-07-10")
season_end = pd.Timestamp("2026-05-31")

df_analysis_final = df_analysis[
    (df_analysis["match_date"] >= season_start) &
    (df_analysis["match_date"] <= season_end)
].copy()

In [67]:
len(df_analysis_final)

225993

In [61]:
df_analysis_final.to_csv("analysis_dataset.csv", index=False)

In [68]:
len(df_analysis)

291814

In [69]:
df_analysis

,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,substituted,team_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender,player_name_clean,player_aiff_id_cleaned,appearance
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Kabir Mehta,418848,True
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,True,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Tavish Sharma,398082,True
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Eklavya Choudhary,522296,False
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Swastik Sharma,398260,True
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,2026-06-19,\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,False,Rajasthan Football School,State,Rajasthan,Youth,Club League,Male,Navya Mittal,418674,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
806269,72239,2044,"Assam Youth League U-15, 2025-26",2026-06-23,SAI field,,True,This match has finished,2,Jyotisman Boro,...,False,SAI Guwahati,State,Assam,Youth,Club League,Male,Jyotisman Boro,424377,False
806270,72239,2044,"Assam Youth League U-15, 2025-26",2026-06-23,SAI field,,True,This match has finished,2,Samir Basfor,...,False,SAI Guwahati,State,Assam,Youth,Club League,Male,Samir Basfor,472713,False
806271,72239,2044,"Assam Youth League U-15, 2025-26",2026-06-23,SAI field,,True,This match has finished,2,Arup Karmakar,...,False,SAI Guwahati,State,Assam,Youth,Club League,Male,Arup Karmakar,322854,False
806272,72239,2044,"Assam Youth League U-15, 2025-26",2026-06-23,SAI field,,True,This match has finished,2,Hintu Ray,...,False,SAI Guwahati,State,Assam,Youth,Club League,Male,Hintu Ray,467164,False


In [73]:
df_analysis['AI_state'].value_counts()

AI_state
National            99871
West Bengal         31052
Goa                 29007
Maharashtra         24854
Karnataka           22811
Gujarat             16875
Delhi               12956
Kerala              11931
Rajasthan           10368
Himachal Pradesh     9250
Puducherry           7645
Assam                6886
Madhya Pradesh       2843
Punjab               1694
Chhattisgarh         1382
Odisha               1319
Manipur              1029
Tamil Nadu             41
Name: count, dtype: int64

In [77]:
club_players = (
    df_analysis[df_analysis["appearance"]]
    .groupby("team_name")
    .agg(
        players_appeared=("player_aiff_id_cleaned", "nunique")
    )
    .sort_values("players_appeared", ascending=False)
    .reset_index()
)
pd.set_option("display.max_rows", None)
club_players.head(100)

,team_name,players_appeared
0,Alchemy International FA,239
1,Kickstart FC Karnataka,208
2,PFC Kerala,182
3,United Sports Club,174
4,East Bengal FC,172
5,BIDHANNAGAR MUNICIPAL SPORTS ACADEMY,161
6,Kerala Blasters Football Club,161
7,Roots Football Club,154
8,Bhawanipore FC,150
9,South United FC,147
